In [1]:
!nvidia-smi

Sun Nov 22 16:47:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!mkdir -p /content/data/tree-counting
!unzip -q /content/drive/MyDrive/tree-counter/tree-counting.zip -d /content/data/tree-counting
!ls /content/data/tree-counting


data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [6]:
!git clone https://github.com/sabeena0/tree-counter.git /content/tree-counter
%cd /content/tree-counter
!pip install -q ultralytics sahi

Cloning into '/content/tree-counter'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 15 (delta 3), reused 13 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), done.
Resolving deltas: 100% (3/3), done.
/content/tree-counter


In [7]:
!python src/check_dataset.py --root /content/data/tree-counting

data.yaml: nc=1  names=['tree']

[train] images=219  boxes=18465  avg/img=84.3  min=16  max=253
         missing labels=0  empty labels=0  malformed=0  polygon lines=0
         classes={0: 18465}  image sizes={(640, 640): 219}
[valid] images=27  boxes=1764  avg/img=65.3  min=7  max=120
         missing labels=0  empty labels=0  malformed=0  polygon lines=0
         classes={0: 1764}  image sizes={(640, 640): 27}
[test] images=27  boxes=2349  avg/img=87.0  min=48  max=130
         missing labels=0  empty labels=0  malformed=0  polygon lines=0
         classes={0: 2349}  image sizes={(640, 640): 27}


In [8]:
yaml_text = """path: /content/data/tree-counting
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['tree']
"""
open('/content/tree.yaml', 'w').write(yaml_text)
print(open('/content/tree.yaml').read())

path: /content/data/tree-counting
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['tree']



In [9]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data='/content/tree.yaml',
    epochs=3,
    imgsz=640,
    batch=16,
    max_det=1000,
    flipud=0.5, fliplr=0.5, degrees=0,
    project='/content/drive/MyDrive/tree-counter/runs',
    name='smoke_test',
    seed=0,
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tree.yaml, degrees=0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hs

In [10]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data='/content/tree.yaml',
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    max_det=1000,
    flipud=0.5, fliplr=0.5, degrees=0,
    project='/content/drive/MyDrive/tree-counter/runs',
    name='baseline_yolov8s',
    seed=0,
)


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tree.yaml, degrees=0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=1000, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_yolov8s, nbs=64, nms=None, opset=None, optimize=False, optimi

In [11]:
from ultralytics import YOLO

best = YOLO('/content/drive/MyDrive/tree-counter/runs/baseline_yolov8s/weights/best.pt')
m = best.val(data='/content/tree.yaml', split='test', max_det=1000, plots=True,
             project='/content/drive/MyDrive/tree-counter/runs', name='baseline_test')
print(f"TEST  P={m.box.mp:.3f}  R={m.box.mr:.3f}  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 40.7±22.5 MB/s, size: 84.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/data/tree-counting/test/labels... 27 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 27/27 581.8it/s 0.0s
val: New cache created: /content/data/tree-counting/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.4it/s 1.5s
                   all         27       2349      0.897      0.817      0.876      0.395
Speed: 3.4ms preprocess, 11.9ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to /content/drive/MyDrive/tree-counter/runs/baseline_test
TEST  P=0.897  R=0.817  mA

In [12]:
from pathlib import Path
import numpy as np

img_dir = Path('/content/data/tree-counting/test/images')
lbl_dir = Path('/content/data/tree-counting/test/labels')
files = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})

preds = best.predict([str(f) for f in files], conf=0.25, iou=0.7, max_det=1000, verbose=False)
true = np.array([sum(1 for l in open(lbl_dir / (f.stem + '.txt')) if l.strip()) for f in files])
pred = np.array([len(p.boxes) for p in preds])

print(f"Images: {len(files)}")
print(f"Total true trees: {true.sum()}   Total predicted: {pred.sum()}   "
      f"Bias: {(pred.sum() - true.sum()) / true.sum() * 100:+.1f}%")
print(f"Mean absolute error per image: {np.abs(true - pred).mean():.1f} trees")
print(f"Mean absolute % error per image: {(np.abs(true - pred) / true).mean() * 100:.1f}%")
for f, t, p in list(zip(files, true, pred))[:8]:
    print(f"{f.name[:32]:34s} true={t:4d} pred={p:4d}")

Images: 27
Total true trees: 2349   Total predicted: 2502   Bias: +6.5%
Mean absolute error per image: 7.4 trees
Mean absolute % error per image: 9.2%
cd_261945_4_174267_4_19_jpg.rf.9   true=  48 pred=  51
cd_261945_4_174267_4_19_jpg.rf.d   true=  48 pred=  51
cd_261945_4_174267_4_19_jpg.rf.e   true=  48 pred=  51
cl_261906_4_174459_4_19_jpg.rf.0   true=  69 pred=  83
cl_261906_4_174459_4_19_jpg.rf.2   true=  69 pred=  83
cl_261906_4_174459_4_19_jpg.rf.8   true=  69 pred=  83
cr_261969_4_174260_4_19_jpg.rf.2   true= 114 pred= 111
cr_261969_4_174260_4_19_jpg.rf.9   true= 114 pred= 111


In [14]:
import re
from pathlib import Path
from collections import Counter

root = Path('/content/data/tree-counting')

def source_ids(split):
    names = [p.name for p in (root / split / 'images').iterdir()]
    return [re.split(r'_(?:jpg|jpeg|png)\.rf\.', n)[0] for n in names]

ids = {s: source_ids(s) for s in ('train', 'valid', 'test')}
sets = {s: set(v) for s, v in ids.items()}

for s in ids:
    print(f"{s:6s} images={len(ids[s]):4d}  unique sources={len(sets[s]):4d}  "
          f"max copies of one source={max(Counter(ids[s]).values())}")

print("\nShared sources between splits:")
print("  train & valid:", len(sets['train'] & sets['valid']))
print("  train & test :", len(sets['train'] & sets['test']))
print("  valid & test :", len(sets['valid'] & sets['test']))

train  images= 219  unique sources=  73  max copies of one source=3
valid  images=  27  unique sources=   9  max copies of one source=3
test   images=  27  unique sources=   9  max copies of one source=3

Shared sources between splits:
  train & valid: 0
  train & test : 0
  valid & test : 0
